# 07.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/03-smoothing-holt-winters/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

y = [100, 120, 110, 130, 125]      # £k, five weeks

def ses_by_hand(y, alpha):
    level = y[0]                    # start the level at the first value
    rows = []
    for t in range(1, len(y)):
        forecast = level            # forecast for week t is the level so far
        level = alpha * y[t] + (1 - alpha) * level
        rows.append({"week": t + 1, "actual": y[t], "forecast": forecast,
                     "error": y[t] - forecast, "new level": level})
    return pd.DataFrame(rows)

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

def ses(train, h):
    return SimpleExpSmoothing(train, initialization_method="estimated").fit().forecast(h).values

def holt(train, h):
    return ExponentialSmoothing(train, trend="add", damped_trend=True,
                                initialization_method="estimated").fit().forecast(h).values

from statsmodels.tsa.holtwinters import ExponentialSmoothing

def holt_winters(train, h):
    """Additive Holt-Winters, seeded with the first year as the starting seasonal pattern."""
    first = train.iloc[:52]
    model = ExponentialSmoothing(
        train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=52,
        initialization_method="known", initial_level=first.mean(), initial_trend=0.0,
        initial_seasonal=(first - first.mean()).values).fit()
    return model.forecast(h).values

768,341 order lines, 2009-12-01 to 2011-12-04
734 days, 105 weeks, 24 months


## Exercise 1

- Week 2: forecast 50, level = 0.3 × 60 + 0.7 × 50 = 53.
- Week 3: forecast 53, level = 0.3 × 55 + 0.7 × 53 = 53.6.
- Week 4: forecast 53.6, level = 0.3 × 70 + 0.7 × 53.6 = 58.52.
- Forecast for the fifth value: **58.52**.

In [2]:
ses_by_hand([50, 60, 55, 70], alpha=0.3).round(2)

,week,actual,forecast,error,new level
0,2,60,50.0,10.0,53.00
1,3,55,53.0,2.0,53.60
2,4,70,53.6,16.4,58.52


## Exercise 2

In [3]:
train = weekly[weekly.index <= "2011-10-16"]
for a in [0.1, 0.3, 0.5, 0.7, 0.9]:
    f = SimpleExpSmoothing(train, initialization_method="estimated").fit(smoothing_level=a, optimized=False)
    print(f"alpha {a}: in-sample MAE £{np.abs(train - f.fittedvalues).mean():,.0f}")
best = SimpleExpSmoothing(train, initialization_method="estimated").fit()
print(f"statsmodels chose alpha = {best.params['smoothing_level']:.3f}")

alpha 0.1: in-sample MAE £38,571
alpha 0.3: in-sample MAE £33,997
alpha 0.5: in-sample MAE £34,805
alpha 0.7: in-sample MAE £36,135
alpha 0.9: in-sample MAE £37,299
statsmodels chose alpha = 0.586


The best in-sample MAE is at α = 0.3, but statsmodels chose α ≈ 0.59. The difference is the loss: statsmodels minimises **squared** errors, which weight the big seasonal jumps (September, Christmas) heavily, and a faster-reacting level catches up with those jumps sooner. Choose the loss that matches how you will be judged: if buyers care about typical weekly error in pounds, MAE is the better guide. Either way, a middling α wins: weekly revenue is persistent, but last week is not all you need.

## Exercise 3

In [4]:
def holt_undamped(train, h):
    return ExponentialSmoothing(train, trend="add", initialization_method="estimated").fit().forecast(h).values

print(f"damped:   £{backtest(holt)['MAE'].mean():,.0f}")
print(f"undamped: £{backtest(holt_undamped)['MAE'].mean():,.0f}")
tr = weekly[weekly.index <= "2011-10-16"]
print("undamped forecast from 16 Oct (£k):", (holt_undamped(tr, 5) / 1000).round(0))
print("damped forecast from 16 Oct (£k):  ", (holt(tr, 5) / 1000).round(0))

damped:   £38,108
undamped: £37,647
undamped forecast from 16 Oct (£k): [230. 230. 231. 231. 232.]
damped forecast from 16 Oct (£k):   [230. 230. 230. 230. 230.]


On ShopWave the two are almost identical, and undamped is even marginally better (by about £500 a week). The reason is in the fitted parameters: β ≈ 0, so there is hardly any trend to damp. Damping matters on series with a real but temporary slope, such as a product launch: undamped Holt would extend the launch growth forever. It is the safer default for business series, even though here it makes no difference.

## Exercise 4

In [5]:
mtrain = monthly[monthly.index <= "2011-10-31"]
first = mtrain.iloc[:12]
f = ExponentialSmoothing(mtrain, trend="add", damped_trend=True, seasonal="mul", seasonal_periods=12,
                         initialization_method="known", initial_level=first.mean(), initial_trend=0.0,
                         initial_seasonal=(first / first.mean()).values).fit()
pred = f.forecast(1).iloc[0]; actual = monthly.loc["2011-11-30"]
print(f"November 2011 forecast £{pred:,.0f}, actual £{actual:,.0f}, error {100 * (pred - actual) / actual:+.1f}%")

November 2011 forecast £1,220,959, actual £1,156,206, error +5.6%


The monthly multiplicative model lands close to the real November. Monthly data is easier: twelve seasonal values instead of fifty-two, no zero months, and the multiplicative form is allowed. The price is coarser timing: it cannot tell the buyers *which week* of November to expect the peak.